# 01 — Introduction to LLMs for Research (Hands‑on)

In this notebook you will:

- Make your first **LLM chat call** (using Groq’s OpenAI‑compatible API).
- Learn the **message format** (`system`, `user`, `assistant`) used throughout the workshop.
- Use **two helper functions** (`chat()` and `chat_reset()`) so later notebooks stay clean and function‑based.

> **Workshop rule:** We’ll call the model through helper functions.  
> You *can* look at the underlying API call, but you don’t need to.


## 1) Imports

We keep imports minimal and explicit so it’s clear what each library is used for.

In [ ]:
# remove # from the line below and run if packages are not installed
# !pip install numpy pandas matplotlib openai requests

In [ ]:
# Standard library: access environment variables safely
import os  # used for storing API keys and configuration

# OpenAI Python client (works with Groq because Groq exposes an OpenAI-compatible endpoint)
from openai import OpenAI  # used to create a client and call chat completions


## 2) Configuration (API key + model)

⚠️ **Keep your API key private.**  
In a real project, it’s better to set keys as environment variables (not in a notebook).  

For this workshop, we keep the pattern simple and consistent with the existing course.

In [ ]:
# Paste your Groq API key below (between quotes).
# NOTE: This is not a secure way of entering an API key because it becomes visible in the notebook.
# For safer setups, prefer environment variables or secret managers (especially if sharing notebooks).

os.environ["GROQ_API_KEY"] = ""  # <-- replace with your key.
model = "llama-3.3-70b-versatile"  # Select a model: https://console.groq.com/docs/models
os.environ["BASE_URL"] = "https://api.groq.com/openai/v1"  # Groq uses an OpenAI-compatible API base URL.


## 3) Create a client

The `OpenAI` client can point to different providers as long as they implement the OpenAI API.
Here we point it to Groq’s base URL and authenticate using your key.

In [ ]:
# Read configuration from the environment (so it’s in one place).
# If the key is missing, calls will fail with an authentication error.
api_key = os.environ["GROQ_API_KEY"]  # your Groq API key (string)
base_url = os.environ["BASE_URL"]     # Groq's OpenAI-compatible endpoint base URL

# Create a client instance.
# This object will be reused for the rest of the workshop.
client = OpenAI(
    api_key=api_key,   # authentication
    base_url=base_url  # provider endpoint
)

# Quick sanity check: print a short confirmation message.
print("✅ Client initialized")


✅ Client initialized


## 4) Your first chat call

A chat call is a list of **messages**. Each message is a dictionary with:

- `role`: `"system"`, `"user"`, or `"assistant"`
- `content`: the text for that message

The model reads the messages and returns the assistant’s next reply.

In [ ]:
# Build a minimal message list.
# - The system message sets the style/rules.
# - The user message is your question/instruction.
messages = [
    {"role": "system", "content": "You are a helpful research assistant. Be concise."},  # behavior
    {"role": "user", "content": "Explain what a large language model (LLM) is in two sentences."},  # task
]

# Call the chat completion endpoint.
response = client.chat.completions.create(
    model=model,        # which model to use
    messages=messages,  # conversation so far
    temperature=0.2,    # low randomness for factual, stable answers
)

# Extract the assistant text safely (this is the part you usually want).
reply_text = response.choices[0].message.content  # the generated assistant reply
print(reply_text)  # display the result


A large language model (LLM) is a type of artificial intelligence (AI) designed to process and understand human language, typically trained on vast amounts of text data to generate human-like responses. LLMs use complex algorithms and neural networks to learn patterns and relationships in language, enabling them to perform tasks such as language translation, text summarization, and conversation generation.


## 5) (Optional) Inspect the response object

Sometimes you want to debug:
- which model was used
- token usage / metadata (if provided)
- full structured payload

In [ ]:
# Many modern OpenAI-style clients support `model_dump()` (Pydantic-style).
# If it's not available, we fall back to casting to a plain dict (best-effort).
payload = response.model_dump() if hasattr(response, "model_dump") else dict(response)  # safe conversion

# Print top-level keys so you know what's inside without dumping everything.
print("Top-level keys:", list(payload.keys()))  # quick peek

# Print the reported model name if available.
print("Model used:", payload.get("model", "unknown"))  # model identifier (string)

# Print token usage if the provider returns it (not all providers do).
usage = payload.get("usage", None)  # usage metadata (may be missing)
print("Usage:", usage)  # token counts, etc. (or None)


Top-level keys: ['id', 'choices', 'created', 'model', 'object', 'service_tier', 'system_fingerprint', 'usage', 'usage_breakdown', 'x_groq']
Model used: llama-3.3-70b-versatile
Usage: {'completion_tokens': 75, 'prompt_tokens': 61, 'total_tokens': 136, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'queue_time': 0.010413828, 'prompt_time': 0.002431792, 'completion_time': 0.151746588, 'total_time': 0.15417838}


## 6) Helper functions (the workshop style)

From now on we’ll use helper functions so:

- notebooks stay clean
- you focus on *research tasks*, not API details
- later notebooks can reuse the same interface

We provide two helpers:

1. `chat(...)` — **stateful**, keeps a short conversation history in this notebook
2. `chat_reset(...)` — deletes history and sets system prompt

This keeps things simple: participants only need to call `chat()` without worrying about API details.

In [ ]:


# Create the client using the API key and base URL.
client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],  # Read the key from the environment variable
    base_url=os.environ["BASE_URL"],     # Read the base URL from the environment variable
)

# We keep a small conversation history in memory to make multi-turn interactions easy.
chat_history = []

def chat_reset(system_prompt="You are a helpful research assistant."):
    """Reset chat history to a single system message (in-place)."""
    chat_history[:] = [{"role": "system", "content": system_prompt}]

chat_reset()

def chat(user_text, temperature=0.2, max_turns=8):
    """Multi-turn helper that keeps a short conversation history.
    Use this instead of calling the API directly in most workshop cells.
    """
    # Add the user's message to the conversation history.
    chat_history.append({"role": "user", "content": user_text})

    # Keep only the system message plus the most recent turns (to manage context length).
    system = chat_history[:1]  # First message is the system instruction
    recent = chat_history[-(max_turns * 2):]  # Each turn has user+assistant, hence *2
    window = system + recent  # Final message window sent to the model

    # Make the API call.
    resp = client.chat.completions.create(
        model=model,         # Model name
        messages=window,     # Conversation history window
        temperature=temperature,  # Randomness / creativity control
    )

    # Extract the assistant's reply text.
    reply = resp.choices[0].message.content

    # Store the assistant reply so the next call has memory.
    chat_history.append({"role": "assistant", "content": reply})

    # Return just the text (simple and workshop-friendly).
    return reply



print("Setup complete. `client`, `model`, and `chat()` are available.")
print("Testing chat function...")
print("User: Hello LLM")
print("Assistant: " + chat("Heloo LLM"))  # Test the chat function

chat_reset()  # Reset chat history for next cells

Setup complete. `client`, `model`, and `chat()` are available.
Testing chat function...
User: Hello LLM
Assistant: Hello. It's nice to meet you. Is there something I can help you with or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss.


## 8) Quick exercises

### Exercise A — rewrite for clarity
Provide a sentence from your current draft and ask for a clearer version.

### Exercise B — extract structured fields
Give a short abstract and ask for: *aim, methods, datasets, main result, limitation*.

### Exercise C — propose search keywords
Ask the model to propose arXiv/Google Scholar keywords for a research question.


In [ ]:
# Exercise A: rewrite for clarity (replace the text with your own).
text = "This paper introduces a method that can potentially improve performance in some tasks."  # example sentence
prompt = "Rewrite this sentence to be clearer and more specific, without adding new facts:\n\n" + text
print(chat(prompt

))


This paper presents a method that may enhance performance in certain specific tasks.


In [ ]:
# Exercise B: extract structured fields from a short abstract-like paragraph.
abstract = (
    "We propose a new approach to classify cell types from single-cell RNA sequencing data using self-supervised learning. "
    "Across three public datasets, our method improves macro-F1 by 3–5% compared to strong baselines. "
    "However, performance drops on rare cell populations with limited labels."
)  # small example text

prompt = (
    "Extract the following fields as JSON with keys:\n"
    "- aim\n- methods\n- datasets\n- main_result\n- limitation\n\n"
    "Text:\n" + abstract
)  # prompt that requests structured output

print(chat(prompt))  # run extraction


Here are the extracted fields as JSON:

```
{
  "aim": "classify cell types from single-cell RNA sequencing data using self-supervised learning",
  "methods": "self-supervised learning",
  "datasets": "three public datasets",
  "main_result": "improves macro-F1 by 3–5% compared to strong baselines",
  "limitation": "performance drops on rare cell populations with limited labels"
}
```


In [ ]:
# Exercise C: generate search keywords for a literature review.
question = "How are LLMs used to accelerate hypothesis generation in biomedical research?"  # example topic

prompt = """Propose 8–12 search keywords/phrases for arXiv and Google Scholar.
    Group them into 3 themes and keep them concise.\n\n"
    Question: """ + question

print(chat(prompt

))


Here are 10 search keywords/phrases grouped into 3 themes:

**Theme 1: LLMs in Biomedical Research**
1. "LLMs in biomedicine"
2. "biomedical hypothesis generation"
3. "large language models biomed"

**Theme 2: Hypothesis Generation and Acceleration**
4. "accelerating hypothesis generation"
5. "biomedical hypothesis formation"
6. "AI-driven hypothesis generation"

**Theme 3: LLM Applications and Techniques**
7. "LLM-based biomedical research tools"
8. "natural language processing biomed"
9. "language models for biomedical discovery"
10. "biomedical text analysis with LLMs"


## 9) Wrap-up

You now have:

- A working Groq/OpenAI-compatible client
- The chat **message format** (`system`, `user`, `assistant`)
- Two helper functions used in later notebooks:
  - `chat(...)` for iterative tasks with short memory
  - `chat_reset(...)` to restart chat and set system prompt

